# 365 Probabilidades - Dia #130
## Qual a probabilidade de uma casa no Brasil ainda não ter banheiro?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-21
**Ferramenta:** Python
**Decisão analisada:** O que a gente banalizou já chegou a todo mundo?
**Hashtag:** #365Probabilidades #Dia130

---

### 📖 A História

Um leitor comentou o Dia #120, o da pia que era sonho e virou bancada. Ele contou que
entrevistou os avós para entender o que foi ser criança nos anos 1940, e pediu mais dias
sobre as mudanças na vida da população. E fechou com uma lista do que ele valorizava,
não podia ter, e hoje banaliza: a fotografia, o ar-condicionado, a fruta de outro país.

Comecei pelo item que não estava na lista dele, porque ninguém põe na lista. O banheiro.

Em 1940, menos de 20% dos lares brasileiros tinham um. A pergunta de hoje é quantos
ainda não têm.

---

### 📚 O Conceito

O Censo pergunta a cada domicílio do país se há banheiro, quantos são e de quem é o
uso. Banheiro, para o IBGE, é o cômodo com vaso sanitário e chuveiro ou ducha. A casa
pode ter um banheiro só dela, um banheiro dividido com outra casa, só um sanitário ou
buraco no terreno, ou nada.

Como o Censo é universo, e não amostra, a proporção que ele publica não tem intervalo
de confiança. É a resposta. O modelo próprio deste dia faz outra coisa com ela: compara
três censos para medir a velocidade da mudança, pergunta de quem é a casa sem banheiro
(idade, estado) e traduz a fração em algo que se vê, uma sala de aula.

---

### 🧮 O Modelo

Dia de **probabilidade simples**: a pergunta vem primeiro, e o dado é contagem.

**Fontes (todas IBGE, Censo Demográfico, resultados do universo):**
- **Censo 2022**, tabela SIDRA 6806 (domicílios por existência de banheiro, Brasil e
  unidades da federação) e tabela 9397 (moradores por existência de banheiro e grupo de
  idade). 72.456.368 domicílios; 202.083.020 moradores.
- **Censo 2010**, tabela SIDRA 1395: 57.324.167 domicílios por existência de banheiro ou
  sanitário.
- **Censo 2000**, tabela SIDRA 1438: 44.795.101 domicílios e 168.370.893 moradores por
  número de banheiros, urbano e rural.
- **1940**: a apresentação do IBGE sobre as características dos domicílios no Censo 2022
  afirma que o banheiro "não chegava a 20% dos lares" em 1940. Entra como ponto de
  referência do próprio IBGE, não como contagem nossa.
- Os três arquivos ao lado deste notebook (`censo_banheiro_brasil.csv`,
  `censo_banheiro_uf_2022.csv`, `censo2022_moradores_idade.csv`) são as tabelas acima,
  lidas pela API do SIDRA.

**Modelo próprio:**
- Três leituras da pergunta, da mais estrita à mais larga: sem banheiro nem sanitário;
  sem banheiro (só sanitário ou nada); sem banheiro de uso exclusivo.
- A velocidade: ajuste exponencial da fração sem banheiro em 2000, 2010 e 2022, com
  meia-vida e projeção declarada como extrapolação.
- De quem é a casa: fração por grupo de idade e por estado, com razão entre extremos.
- A sala de aula: Monte Carlo (semente 42) e conta exata de 1 − (1 − p)ⁿ para a chance
  de ao menos uma criança da turma morar em casa sem banheiro.
- Sem ×0,80: Censo, registro de universo.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

print("✅ Bibliotecas carregadas")


In [ ]:
# --- OS DADOS ---
def achar(nome):
    cands = [nome, os.path.join("dados", nome), os.path.join("..", "dados", nome)]
    c = next((c for c in cands if os.path.exists(c)), None)
    if c is None:
        raise FileNotFoundError(f"Salve {nome} na mesma pasta deste notebook (pasta atual: {os.getcwd()}).")
    return c

br = pd.read_csv(achar("censo_banheiro_brasil.csv"))
uf = pd.read_csv(achar("censo_banheiro_uf_2022.csv"))
idade = pd.read_csv(achar("censo2022_moradores_idade.csv"))

def v(ano, unidade, categoria, situacao="Total"):
    s = br[(br["ano"] == ano) & (br["unidade"] == unidade) & (br["categoria"] == categoria) & (br["situacao"] == situacao)]
    return float(s["valor"].iloc[0])

# 2022: as três leituras da pergunta
T22 = v(2022, "domicilios", "total")
nada22 = v(2022, "domicilios", "nenhum")
sanit22 = v(2022, "domicilios", "so_sanitario")
comum22 = v(2022, "domicilios", "so_comum")
p_nada = nada22 / T22
p_sem_banheiro = (nada22 + sanit22) / T22
p_sem_exclusivo = (nada22 + sanit22 + comum22) / T22
P22 = v(2022, "moradores", "total")
m_nada = v(2022, "moradores", "nenhum"); m_sanit = v(2022, "moradores", "so_sanitario"); m_comum = v(2022, "moradores", "so_comum")

# 2010 e 2000, na leitura comparável: sem banheiro (só sanitário ou nada)
T10 = v(2010, "domicilios", "total"); sem10 = v(2010, "domicilios", "so_sanitario") + v(2010, "domicilios", "nenhum")
T00 = v(2000, "domicilios", "total"); sem00 = v(2000, "domicilios", "sem_banheiro")
rural00 = v(2000, "domicilios", "sem_banheiro", "Rural") / v(2000, "domicilios", "total", "Rural")
urb00 = v(2000, "domicilios", "sem_banheiro", "Urbana") / v(2000, "domicilios", "total", "Urbana")
mor00 = v(2000, "moradores", "sem_banheiro") / v(2000, "moradores", "total")
serie = pd.DataFrame({"ano": [2000, 2010, 2022], "sem_banheiro": [sem00 / T00, sem10 / T10, p_sem_banheiro],
                      "n": [sem00, sem10, nada22 + sanit22]})
P1940 = 0.20   # teto declarado pelo IBGE ("não chegava a 20% dos lares" tinham banheiro) → sem banheiro > 80%

# banheiros por casa
um10 = v(2010, "domicilios", "exclusivo_1") / v(2010, "domicilios", "exclusivo")
um22 = v(2022, "domicilios", "exclusivo_1") / v(2022, "domicilios", "exclusivo")
dois_mais22 = (v(2022, "domicilios", "exclusivo_2") + v(2022, "domicilios", "exclusivo_3") + v(2022, "domicilios", "exclusivo_4mais")) / T22

# estados e idade
uf["pct_sem_exclusivo"] = uf["pct_comum"] + uf["pct_sanitario"] + uf["pct_nenhum"]
uf = uf.sort_values("pct_sem_exclusivo", ascending=False)
idade["sem_exclusivo"] = idade["so_comum"] + idade["so_sanitario"] + idade["nenhum"]
idade["p_sem_exclusivo"] = idade["sem_exclusivo"] / idade["total"]
idade["p_nada"] = idade["nenhum"] / idade["total"]
cri = idade.iloc[:3].sum(numeric_only=True)          # 0 a 14 anos
p_cri_sem = cri["sem_exclusivo"] / cri["total"]; p_cri_nada = cri["nenhum"] / cri["total"]
vel = idade[idade["faixa"].isin(["65 a 69", "70 a 74", "75 a 79", "80 a 84", "85 a 89", "90 a 94", "95 a 99", "100 ou mais"])].sum(numeric_only=True)
p_vel_sem = vel["sem_exclusivo"] / vel["total"]

print("=" * 72)
print("  OS DADOS - CENSO 2022, 2010, 2000 (IBGE, universo)")
print("=" * 72)
print(f"  2022: {mil(T22)} domicílios, {mil(P22)} moradores")
print(f"    sem banheiro nem sanitário:            {mil(nada22):>10}  {pct(p_nada, 2):>6}   ({mil(m_nada)} pessoas)")
print(f"    só sanitário ou buraco no terreno:     {mil(sanit22):>10}  {pct(sanit22 / T22, 2):>6}   ({mil(m_sanit)} pessoas)")
print(f"    só banheiro dividido com outra casa:   {mil(comum22):>10}  {pct(comum22 / T22, 2):>6}   ({mil(m_comum)} pessoas)")
print(f"    sem banheiro (sanitário ou nada):      {mil(nada22 + sanit22):>10}  {pct(p_sem_banheiro, 2):>6}")
print(f"    sem banheiro de uso exclusivo:         {mil(nada22 + sanit22 + comum22):>10}  {pct(p_sem_exclusivo, 2):>6}   ({mil(m_nada + m_sanit + m_comum)} pessoas)")
print()
print("  A série, na leitura 'sem banheiro' (só sanitário ou nada):")
print(f"    1940: mais de 80% (o IBGE registra banheiro em menos de 20% dos lares)")
for _, r in serie.iterrows():
    print(f"    {int(r['ano'])}: {pct(r['sem_banheiro'])}  ({mil(r['n'])} domicílios)")
print(f"    2000, rural: {pct(rural00)} das casas sem banheiro; urbano: {pct(urb00)}; {pct(mor00)} das pessoas")
print()
print(f"  Banheiros por casa: casas com banheiro e um só, {pct(um10)} em 2010 → {pct(um22)} em 2022; dois ou mais em {pct(dois_mais22)} das casas")
print()
print(f"  Crianças de 0 a 14 anos ({mil(cri['total'])}): sem banheiro exclusivo {pct(p_cri_sem)}; sem nada {pct(p_cri_nada, 2)} ({mil(cri['nenhum'])} crianças)")
print(f"  65 anos ou mais: sem banheiro exclusivo {pct(p_vel_sem)}")
print(f"  Estados, sem banheiro exclusivo: de {vir(uf['pct_sem_exclusivo'].min(), 2)}% ({uf.iloc[-1]['nome']}) a {vir(uf['pct_sem_exclusivo'].max(), 2)}% ({uf.iloc[0]['nome']})")
print("=" * 72)


In [ ]:
# --- O MODELO ---
N_SIM = 200_000

# Camada 1: a velocidade. Queda exponencial da fração sem banheiro, 2000 → 2010 → 2022
anos = serie["ano"].to_numpy(dtype=float); y = np.log(serie["sem_banheiro"].to_numpy())
b1, b0 = np.polyfit(anos - 2000, y, 1)
taxa = -b1                                  # queda proporcional por ano
meia_vida = np.log(2) / taxa
def proj(ano):
    return np.exp(b0 + b1 * (ano - 2000))
ajuste = proj(anos)
ano_meio_pct = 2000 + (np.log(0.005) - b0) / b1       # quando chega a 0,5%
ano_um_por_mil = 2000 + (np.log(0.001) - b0) / b1     # quando chega a 0,1%
# de 1940 a 2000, a velocidade foi outra: de mais de 80% para 16,7% em 60 anos
taxa_1940 = (np.log(0.80) - np.log(serie["sem_banheiro"].iloc[0])) / 60

# Camada 2: de quem é a casa. Razões entre extremos
razao_estado = uf["pct_sem_exclusivo"].max() / uf["pct_sem_exclusivo"].min()
razao_nada = uf["pct_nenhum"].max() / uf["pct_nenhum"].min()
razao_idade = p_cri_sem / p_vel_sem
top5 = uf.head(5); fundo5 = uf.tail(5)

# Camada 3: a sala de aula. Turma de n crianças sorteadas do Brasil
def p_pelo_menos_uma(n, p):
    return 1 - (1 - p) ** n
N_TURMA = 30
p_turma_sem = p_pelo_menos_uma(N_TURMA, p_cri_sem)
p_turma_nada = p_pelo_menos_uma(N_TURMA, p_cri_nada)
turmas = ger.binomial(N_TURMA, p_cri_sem, N_SIM)
mc_turma = np.mean(turmas >= 1); mc_media = turmas.mean()
# a mesma turma nos extremos, usando a fração de domicílios do estado como aproximação declarada
p_turma_pi = p_pelo_menos_uma(N_TURMA, uf[uf["sigla"] == "PI"]["pct_sem_exclusivo"].iloc[0] / 100)
p_turma_ma = p_pelo_menos_uma(N_TURMA, uf[uf["sigla"] == "MA"]["pct_sem_exclusivo"].iloc[0] / 100)
p_turma_sp = p_pelo_menos_uma(N_TURMA, uf[uf["sigla"] == "SP"]["pct_sem_exclusivo"].iloc[0] / 100)
ns = np.arange(1, 61)

print("=" * 72)
print("  O MODELO")
print("=" * 72)
print("  A pergunta, em três leituras (Censo 2022):")
print(f"    sem banheiro nem sanitário:        {pct(p_nada, 2)}   1 casa em {mil(1 / p_nada)}")
print(f"    sem banheiro (sanitário ou nada):  {pct(p_sem_banheiro, 2)}   1 casa em {mil(1 / p_sem_banheiro)}")
print(f"    sem banheiro de uso exclusivo:     {pct(p_sem_exclusivo, 2)}   1 casa em {mil(1 / p_sem_exclusivo)}")
print()
print("  Camada 1 · a velocidade (2000 → 2010 → 2022, leitura 'sem banheiro')")
print(f"    queda de {pct(taxa)} ao ano; a fração cai pela metade a cada {vir(meia_vida)} anos")
print(f"    ajuste: " + "  ".join(f"{int(a)} {pct(f)} (obs {pct(o)})" for a, f, o in zip(anos, ajuste, serie['sem_banheiro'])))
print(f"    se a velocidade se mantiver: 0,5% em {int(round(ano_meio_pct))}, 0,1% em {int(round(ano_um_por_mil))}  (extrapolação declarada)")
print(f"    de 1940 a 2000 a queda foi de {pct(taxa_1940)} ao ano; o ritmo mais que quadruplicou")
print()
print("  Camada 2 · de quem é a casa")
print(f"    por estado (sem banheiro exclusivo): {uf.iloc[0]['nome']} {vir(uf.iloc[0]['pct_sem_exclusivo'], 2)}% contra {uf.iloc[-1]['nome']} {vir(uf.iloc[-1]['pct_sem_exclusivo'], 2)}%, razão de {vir(razao_estado, 0)}×")
print(f"    sem nada: {uf.sort_values('pct_nenhum').iloc[-1]['nome']} {vir(uf['pct_nenhum'].max(), 2)}% contra {vir(uf['pct_nenhum'].min(), 2)}%, razão de {vir(razao_nada, 0)}×")
print(f"    por idade: crianças de 0 a 14 {pct(p_cri_sem)} contra 65 ou mais {pct(p_vel_sem)}, razão de {vir(razao_idade, 1)}×")
print()
print(f"  Camada 3 · a sala de aula ({N_TURMA} crianças sorteadas do Brasil)")
print(f"    P(ao menos uma mora em casa sem banheiro exclusivo) = {pct(p_turma_sem)}   Monte Carlo {pct(mc_turma)}   média {vir(mc_media, 2)} crianças por turma")
print(f"    P(ao menos uma mora em casa sem banheiro nem sanitário) = {pct(p_turma_nada)}")
print(f"    a mesma turma no Piauí {pct(p_turma_pi)}, no Maranhão {pct(p_turma_ma)}, em São Paulo {pct(p_turma_sp)}  (fração de domicílios do estado como aproximação)")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · a série ══
fig1, ax1 = plt.subplots(figsize=(11, 6))
xs = np.arange(1995, 2041)
ax1.plot(xs, proj(xs) * 100, color=CINZA, lw=1.4, ls="--", label=f"ajuste exponencial, meia-vida de {vir(meia_vida)} anos (extrapolação depois de 2022)")
ax1.plot(serie["ano"], serie["sem_banheiro"] * 100, marker="o", ms=11, lw=2.8, color=VERMELHO, label="casas sem banheiro (só sanitário ou nada), Censo")
for _, r in serie.iterrows():
    ax1.text(r["ano"] + 0.6, r["sem_banheiro"] * 100 + 0.9, pct(r["sem_banheiro"]), fontsize=12, fontweight="bold", color=VERMELHO)
ax1.scatter([1940], [80], color=DOURADO, s=110, zorder=5, marker="^")
ax1.text(1944, 80, "1940: mais de 80%\n(IBGE, referência)", fontsize=10, color="#8a6f1f", va="center")
ax1.set_xlim(1935, 2041); ax1.set_ylim(0, 90)
ax1.set_xlabel("ano do Censo"); ax1.set_ylabel("domicílios sem banheiro (%)")
ax1.legend(fontsize=9.5, loc="upper right")
ax1.set_title(f"Em 2000, uma casa em seis não tinha banheiro; em 2022, uma em {mil(1 / p_sem_banheiro)}\n"
              f"a fração cai pela metade a cada {vir(meia_vida)} anos; em 1940 o banheiro era minoria", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, Censos 2000 (SIDRA 1438), 2010 (1395) e 2022 (6806); 1940 pela apresentação do IBGE  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-130-grafico-01-a-serie.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · estado a estado ══
fig2, ax2 = plt.subplots(figsize=(11, 8.4))
u = uf.sort_values("pct_sem_exclusivo")
yy = np.arange(len(u))
ax2.barh(yy, u["pct_nenhum"], color=VERMELHO, alpha=0.95, height=0.72, label="sem banheiro nem sanitário")
ax2.barh(yy, u["pct_sanitario"], left=u["pct_nenhum"], color=DOURADO, alpha=0.95, height=0.72, label="só sanitário ou buraco")
ax2.barh(yy, u["pct_comum"], left=u["pct_nenhum"] + u["pct_sanitario"], color=VERDE, alpha=0.95, height=0.72, label="só banheiro dividido com outra casa")
for i, (_, r) in enumerate(u.iterrows()):
    ax2.text(r["pct_sem_exclusivo"] + 0.15, i, vir(r["pct_sem_exclusivo"], 1) + "%", va="center", fontsize=9.5, color="#333333",
             bbox=dict(facecolor="white", edgecolor="none", pad=0.5), zorder=5)
ax2.axvline(p_sem_exclusivo * 100, color="#333333", lw=1.2, ls="--")
ax2.text(p_sem_exclusivo * 100 + 0.15, 1.5, f"Brasil {pct(p_sem_exclusivo)}", fontsize=10, color="#333333")
ax2.set_yticks(yy); ax2.set_yticklabels(u["nome"], fontsize=10)
ax2.set_xlim(0, 15.5); ax2.set_xlabel("domicílios sem banheiro de uso exclusivo (%)")
ax2.legend(fontsize=9.5, loc="lower right")
ax2.set_title(f"No Acre e no Maranhão, uma casa em sete não tem banheiro só dela; em São Paulo, uma em mil\n"
              "Censo 2022, unidades da federação", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, Censo 2022, tabela SIDRA 6806  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-130-grafico-02-estado-a-estado.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · as crianças ══
fig3, ax3 = plt.subplots(figsize=(11, 6))
xi = np.arange(len(idade))
ax3.bar(xi, idade["p_sem_exclusivo"] * 100, color=[VERMELHO if i < 3 else CINZA for i in xi], alpha=0.9)
ax3.axhline(p_sem_exclusivo * 100, color="#333333", lw=1.2, ls="--")
ax3.text(len(idade) - 0.5, p_sem_exclusivo * 100 + 0.1, f"todas as idades {pct(p_sem_exclusivo)}", ha="right", fontsize=10, color="#333333")
ax3.text(1, p_cri_sem * 100 + 0.35, f"0 a 14 anos: {pct(p_cri_sem)}", ha="center", fontsize=12, fontweight="bold", color=VERMELHO)
ax3.set_xticks(xi); ax3.set_xticklabels(idade["faixa"], rotation=60, ha="right", fontsize=9.5)
ax3.set_ylabel("moradores em casa sem banheiro exclusivo (%)"); ax3.set_ylim(0, 4.6)
ax3.set_title(f"A casa sem banheiro é a casa das crianças: {pct(p_cri_sem)} dos menores de 15 anos, {pct(p_vel_sem)} dos maiores de 65\n"
              f"{mil(cri['sem_exclusivo'])} crianças sem banheiro exclusivo; {mil(cri['nenhum'])} sem banheiro nem sanitário", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, Censo 2022, tabela SIDRA 9397 (moradores por grupo de idade)  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-130-grafico-03-as-criancas.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a sala de aula ══
fig4, ax4 = plt.subplots(figsize=(11, 6))
ax4.plot(ns, p_pelo_menos_uma(ns, p_cri_sem) * 100, color=VERMELHO, lw=2.8, label=f"sem banheiro exclusivo, {pct(p_cri_sem)} das crianças")
ax4.plot(ns, p_pelo_menos_uma(ns, p_cri_nada) * 100, color=DOURADO, lw=2.8, label=f"sem banheiro nem sanitário, {pct(p_cri_nada, 2)} das crianças")
ax4.axvline(N_TURMA, color="#333333", lw=1, ls="--")
ax4.text(N_TURMA + 1.2, p_turma_sem * 100 - 7, pct(p_turma_sem), fontsize=12, fontweight="bold", color=VERMELHO)
ax4.text(N_TURMA + 0.8, p_turma_nada * 100 - 7, pct(p_turma_nada), fontsize=12, fontweight="bold", color="#8a6f1f")
ax4.text(N_TURMA + 0.8, 4, "turma de 30", fontsize=10, color="#333333")
ax4.set_xlabel("crianças na turma"); ax4.set_ylabel("chance de ao menos uma morar em casa sem banheiro (%)")
ax4.set_xlim(0, 61); ax4.set_ylim(0, 100)
ax4.legend(fontsize=10, loc="lower right")
ax4.set_title(f"Numa turma de 30 crianças sorteadas do Brasil, {pct(p_turma_sem, 0)} de chance de ao menos uma não ter banheiro em casa\n"
              f"no Piauí e no Maranhão a mesma turma passa de 90%; em São Paulo fica em {pct(p_turma_sp)}", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Conta própria: 1 − (1 − p)ⁿ sobre a fração de crianças do Censo 2022; turma sorteada ao acaso é hipótese declarada  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-130-grafico-04-a-sala-de-aula.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Uma casa em cada 53 no Brasil não tem banheiro de uso exclusivo: 1,9%, 1,37 milhão de
domicílios, 4,5 milhões de pessoas.** Na leitura mais estrita, sem banheiro nem
sanitário, uma em 197: 367 mil casas, 1,2 milhão de pessoas.

O número é pequeno porque a queda foi enorme. Em 1940, o banheiro era minoria. Em 2000,
uma casa em seis não tinha; no campo, mais da metade. Em 2010, uma em quinze. Em 2022,
uma em 70. A fração cai pela metade a cada 6 anos, e o ritmo é mais de quatro vezes o do
século passado.

Mas 1,9% não é uma média espalhada. É um endereço. No Acre e no Maranhão, uma casa em
sete. Em São Paulo, uma em mil. E é a casa das crianças: 3,5% dos menores de 15 anos
moram sem banheiro exclusivo, contra 1,3% dos maiores de 65. São 1,4 milhão de crianças,
354 mil delas sem banheiro nem sanitário.

Numa turma de 30 crianças sorteadas do país, a chance de ao menos uma voltar para uma
casa sem banheiro é de 65%.

O leitor tinha razão em um ponto: a fotografia, o ar-condicionado e a fruta de outro
país ficaram baratos e a gente banalizou. O banheiro chegou mais longe que todos. E
ainda não chegou em todo mundo.

*Você sabe o ano em que a casa dos seus avós ganhou um banheiro?*

---

### ⚠️ Limitações do Modelo

- As definições mudam entre censos. Em 2000 a pergunta era o número de banheiros; em
  2010 entrou a distinção entre banheiro de uso exclusivo e sanitário; em 2022 entrou o
  banheiro dividido com outra casa. A série usa a leitura comparável (sem banheiro,
  isto é, só sanitário ou nada), e a leitura mais larga de 2022 inclui a categoria nova.
- O ponto de 1940 é a frase do IBGE na apresentação do Censo 2022 ("não chegava a 20%
  dos lares"), tratado como teto, não como contagem nossa. Os censos de 1970 a 1991 não
  entraram por falta de tabela comparável aberta; a meia-vida é calculada só com 2000,
  2010 e 2022.
- O ajuste exponencial tem três pontos. A meia-vida descreve o que aconteceu; a projeção
  para 2030 e depois é extrapolação declarada, e a última milha costuma ser mais lenta.
- A sala de aula sorteia crianças do país inteiro. Turmas reais são de um bairro, e a
  fração varia mais entre bairros do que entre estados. Nos extremos por estado, a
  conta usa a fração de domicílios, não de crianças, porque a tabela por idade e
  estado não foi lida.
- Censo é universo, sem intervalo de confiança; a omissão de cobertura do Censo 2022
  (domicílios não visitados) não está corrigida aqui.
- Banheiro, para o IBGE, é cômodo com vaso e chuveiro. Ter banheiro não diz nada sobre
  esgoto: 62,5% da população tinha rede de esgoto em 2022, assunto para outro dia.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
